<h1>Chapter 1 - RAG Setup</h1><i>Building your first RAG (Retrieval Augmented Generation) system.</i><a>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch01_RAG_intro/rag_basics.ipynb)

---

This notebook is for Chapter 1 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Install Required Packages

In [1]:
%pip install "numpy<2" openai==1.12.0 chromadb==0.4.22 tiktoken==0.5.2 python-dotenv==1.0.0

  Using cached numpy-1.26.4-cp313-cp313-win_amd64.whl
  Using cached openai-1.12.0-py3-none-any.whl.metadata (18 kB)
  Using cached chromadb-0.4.22-py3-none-any.whl.metadata (7.3 kB)
  Using cached tiktoken-0.5.2.tar.gz (32 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached chroma_hnswlib-0.7.3-cp313-cp313-win_amd64.whl
  Using cached pulsar_client-3.10.0-cp313-cp313-win_amd64.whl.metadata (1.2 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached wrapt-1.17.3-cp313-cp313-win_amd64.whl.metadata (6.5 kB)
Using cached openai-1.12.0-py3-none-any.whl (226 kB)
Using cached chromadb-0.4.22-py3-none-any.whl (509 kB)
Using cached wrapt-1.17.3-cp313-cp313-win_amd6

  error: subprocess-exited-with-error
  
  exit code: 1
  
  [50 lines of output]
  C:\Users\z004j58u\AppData\Local\Temp\pip-build-env-89wln6op\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
  !!
  
          ********************************************************************************
          Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
  
          By 2027-Feb-18, you need to update your project and remove deprecated calls
          or your builds will no longer be supported.
  
          See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
          ********************************************************************************
  
  !!
    corresp(dist, value, root_dir)
  running bdist_wheel
  running build
  run

### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [2]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [3]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

⚠ Running locally. Using ../datasets/ directory


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [4]:
import os

if IN_COLAB:
    from google.colab import userdata  # type: ignore

    api_key = userdata.get("OPENAI_API_KEY")

    if not api_key:
        raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

    os.environ["OPENAI_API_KEY"] = api_key

## Step 1: Text Chunking

Split documents into smaller chunks with overlap to preserve context.

In [5]:
from pathlib import Path
import chromadb
from openai import OpenAI


def chunk_text(text, size=1000, overlap=200):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            bp = text.rfind("\n\n", start, end)
            if bp == -1:
                bp = text.rfind(". ", start, end)
            if bp > start:
                end = bp + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap if end < len(text) else end
    return chunks


base_datasets_dir = Path("/content/datasets") if IN_COLAB else Path("../datasets")
file_path = base_datasets_dir / "text_files" / "harry_potter_knowledge_base.txt"
if not file_path.exists():
    file_path = Path("./datasets/text_files/harry_potter_knowledge_base.txt")

text = file_path.read_text(encoding="utf-8")
chunks = chunk_text(text)

## Step 2: Embed and Store in ChromaDB

Generate embeddings and store them in a vector database.

In [6]:
import httpx # Add this import
client = OpenAI(http_client=httpx.Client())
embedding_model = "text-embedding-3-small"


def embed_and_store(chunks, db_path, collection_name):
    chroma = chromadb.PersistentClient(path=str(db_path))
    collection = chroma.get_or_create_collection(
        name=collection_name,
        metadata={"description": "Harry Potter knowledge base"},
    )

    embeddings = []
    for i in range(0, len(chunks), 100):
        batch = chunks[i : i + 100]
        res = client.embeddings.create(model=embedding_model, input=batch)
        embeddings.extend([x.embedding for x in res.data])

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=embeddings,
        metadatas=[{"chunk_index": i} for i in range(len(chunks))],
    )
    return collection


chroma_db_dir = Path("chroma_db")
collection = embed_and_store(chunks, chroma_db_dir, "harry_potter_kb")
collection

Collection(name=harry_potter_kb)

## Step 3: Test Retrieval

Test that the retrieval system finds relevant chunks for a given question.

In [7]:
def retrieve(question, top_k=3):
    q_emb = client.embeddings.create(
        model=embedding_model,
        input=question,
    ).data[0].embedding

    res = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents"],
    )

    return res["documents"][0]


question = "Why did Uncle Vernon take the family to a hut in the middle of the sea?"
docs = retrieve(question)
docs

["dozen, then by the hundred. They came through the chimney, through cracks in the door, even delivered by owls. Uncle Vernon tried everything to stop them, but the letters found Harry wherever he was.\n\nFinally, in a desperate attempt to escape the letters, Uncle Vernon drove the family to a hut on a rock in the middle of the sea. On the stroke of midnight on Harry's eleventh birthday, there was a tremendous bang on the door, and it was blasted open by Hagrid, the giant gamekeeper from Hogwarts. Hagrid was furious to discover that the Dursleys had never told Harry about his magical heritage, about his parents' true fate, or about his acceptance to Hogwarts School of Witchcraft and Wizardry.",
 "Little Whinging, Surrey. There was no stamp, and when Harry turned it over, he saw a purple wax seal bearing a coat of arms with a lion, an eagle, a badger, and a snake surrounding a large letter H.\n\nBefore Harry could open it, Uncle Vernon snatched it away. After reading it, his face went f

## Step 4: Test Generation

Feed the retrieved documents and question to the LLM to generate an answer.

In [8]:
def answer(question, docs):
    context = "\n\n---\n\n".join(docs)
    prompt = f"""Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:"""

    res = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
    )

    return res.choices[0].message.content


question = "Why did Uncle Vernon take the family to a hut in the middle of the sea?"
docs = retrieve(question)
answer_text = answer(question, docs)
answer_text

'He drove them there in a desperate attempt to escape the letters that kept arriving for Harry—letters from Hogwarts about his acceptance and his magical heritage.'